In [ ]:
#detecting if active or no 

import pathlib
import platform
import torch
import cv2
import numpy as np
from collections import deque

# 1. System Fixes
if platform.system() == 'Windows':
    pathlib.PosixPath = pathlib.WindowsPath

# 2. Load Model
model = torch.hub.load('ultralytics/yolov5', 'custom', path='C:/Users/AMIT/Desktop/cvvvv/best.pt')
if torch.cuda.is_available():
    model.cuda()

# 3. Settings & Logic Parameters
video_path = 'C:/Users/AMIT/Desktop/cvvvv/video2.mp4'
cap = cv2.VideoCapture(video_path)
FPS = int(cap.get(cv2.CAP_PROP_FPS)) or 30 

# --- UPDATED LOGIC CONSTANTS ---
BUFFER_SIZE = 15      # Increased: Compares current frame to 15 frames ago (0.5s gap)
SENSITIVITY = 10      # Increased: Ignore pixel "vibrations" under 10px
CONF_THRESHOLD = 0.5  # New: Only process detections with > 50% confidence
IDLE_WAIT_SECONDS = 1
IDLE_FRAME_LIMIT = FPS * IDLE_WAIT_SECONDS 

# State Tracking
history = {}       
idle_counters = {} 
current_status = {} 

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    frame = cv2.resize(frame, (640, 480))
    results = model(frame)
    df = results.pandas().xyxy[0] 

    for _, row in df.iterrows():
        # --- NEW: CONFIDENCE FILTER ---
        if row['confidence'] < CONF_THRESHOLD:
            continue

        label = row['name']
        x1, y1, x2, y2 = row['xmin'], row['ymin'], row['xmax'], row['ymax']
        current_box = np.array([x1, y1, x2, y2])

        # Initialize tracking for new objects
        if label not in history:
            history[label] = deque(maxlen=BUFFER_SIZE)
            idle_counters[label] = 0
            current_status[label] = "Initializing..."

        # --- STEP 1: CALCULATE INSTANT MOVEMENT ---
        is_moving_now = False
        if len(history[label]) == BUFFER_SIZE:
            old_box = history[label][0]
            # np.max catches if ANY corner (arm or body) moves significantly
            movement = np.max(np.abs(current_box - old_box))
            
            if movement > SENSITIVITY:
                is_moving_now = True

        # --- STEP 2: APPLY TEMPORAL RULE ---
        if is_moving_now:
            idle_counters[label] = 0
            current_status[label] = "ACTIVE"
        else:
            idle_counters[label] += 1
            if idle_counters[label] >= IDLE_FRAME_LIMIT:
                current_status[label] = "IDLE"

        history[label].append(current_box)

        # --- STEP 3: VISUALIZATION ---
        status = current_status[label]
        color = (0, 255, 0) if status == "ACTIVE" else (0, 0, 255)
        if status == "Initializing...": color = (200, 200, 200)

        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        # Added confidence score to the label for your debugging
        text = f"{label} ({row['confidence']:.2f}): {status}"
        cv2.putText(frame, text, (int(x1), int(y1) - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.imshow("Construction Monitor (Filtered)", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

requirements: Ultralytics requirement ['setuptools>=70.0.0'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command 'pip install --no-cache-dir "setuptools>=70.0.0" ' returned non-zero exit status 1.


Using cache found in C:\Users\AMIT/.cache\torch\hub\ultralytics_yolov5_master


WARNING Retry 2/2 failed: Command 'pip install --no-cache-dir "setuptools>=70.0.0" ' returned non-zero exit status 1.
WARNING requirements:  Command 'pip install --no-cache-dir "setuptools>=70.0.0" ' returned non-zero exit status 1.


YOLOv5  2026-4-5 Python-3.11.9 torch-2.8.0+cpu CPU

Fusing layers... 
Model summary: 157 layers, 7018216 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 
C:\Users\AMIT/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\AMIT/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\AMIT/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\AMIT/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:899: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprec